# Entrenamiento con datos reales: YOLO26n
**Opción principal: YOLO26.** Para repetir la misma ruta con YOLO11,
cambia únicamente `FAMILY` en la configuración.

- PlantVillage: clasificación de tres clases (tardío, temprano, sana).
- PlantDoc: detección de hojas completas con dos enfermedades. Sus cajas
  **no delimitan lesiones** y no permiten medir severidad por área.
- Ambos entrenamientos son independientes y no incluyen insectos.
- Son pilotos públicos: requieren revisión agronómica y un test local independiente.
- Los checkpoints iniciales son preentrenados generales; el entrenamiento
  es lo que adapta cada modelo al conjunto de papa.

## 1. Proyecto y dependencias
En Colab, selecciona GPU para entrenar. La raíz debe contener `scripts/`,
`requirements-vision.txt` y los cuadernos, además de los datasets.
Puedes definir `IOTTESIS_ROOT` antes de esta celda para usar otra ubicación.
La instalación no cambia deliberadamente la versión CUDA/PyTorch de Colab.

In [1]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')
    os.environ.setdefault('IOTTESIS_ROOT', '/content/drive/MyDrive/Tesis_IOT_Papa')

candidatas = ([Path(os.environ['IOTTESIS_ROOT'])] if os.environ.get('IOTTESIS_ROOT') else [])
candidatas += [Path.cwd(), *Path.cwd().parents, Path('/content/IOTtesis')]
PROJECT_ROOT = next((p.resolve() for p in candidatas
                     if (p / 'scripts/entrenar_yolo26.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Sube/clona el proyecto actualizado y configura IOTTESIS_ROOT con su carpeta raíz.')
sys.path.insert(0, str(PROJECT_ROOT))
print('Proyecto:', PROJECT_ROOT)


MessageError: Failed to issue request POST https://colab.pa.googleapis.com/v1/credential-propagation:enable: Bad Request
Response body: {
  "error": {
    "code": 400,
    "message": "Precondition check failed.",
    "status": "FAILED_PRECONDITION"
  }
}


In [ ]:
%pip install -q -r "{PROJECT_ROOT / 'requirements-vision.txt'}"


In [ ]:
import json
import torch
import ultralytics
from scripts.entrenar_yolo26 import training_plan, train_experiment, evaluate_test, benchmark_cpu

print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if tuple(int(v) for v in ultralytics.__version__.split('.')[:2]) < (8, 4):
    raise RuntimeError('Reinicia la sesión después de instalar Ultralytics compatible con YOLO26.')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('Sin GPU: los entrenamientos completos pueden ser lentos; no son necesarios para verificar el plan.')


## 2. Preparación y revisión de datos reales
El preparador conserva manifiestos, revisiones y particiones. Si las salidas
existen, las verifica sin volver a descargar ni eliminar los datos.
No usa los dibujos sintéticos de `data/dataset_yolo`.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/preparar_datasets_reales.py'),
                '--root', str(PROJECT_ROOT), '--source', 'all'], check=True)
for folder in ['papa_plantvillage_clasificacion', 'papa_plantdoc_yolo']:
    report = json.loads((PROJECT_ROOT / 'data/processed' / folder / 'report.json').read_text())
    print(report['source'], report['images'], report['images_per_class'])


## 3. Configuración común
Solo las siguientes celdas de entrenamiento consumen GPU. Verificar los
planes no entrena ni descarga pesos. Mantén test reservado hasta fijar
la configuración con validación.

In [ ]:
FAMILY = 'yolo26'  # Alternativas: 'yolo11', 'yolov8'
EPOCHS = 50
planes = {source: training_plan(PROJECT_ROOT, source, FAMILY, epochs=EPOCHS)
          for source in ['plantvillage', 'plantdoc']}
for plan in planes.values():
    print(plan['source'], plan['initial_weights'], plan['task'], plan['images'], 'imágenes')


## A. Clasificación PlantVillage
YOLO26n-cls aprende el orden de clases desde las carpetas. Consulta los
nombres del modelo: no supongas que coinciden con los IDs del detector.
La clase sana tiene menos imágenes; revisar también resultados por clase.

In [ ]:
clasificacion = train_experiment(planes['plantvillage'], device=DEVICE)
print('Validación:', clasificacion['validation'])
print('Clases:', clasificacion['names'])
print('Mejor checkpoint:', clasificacion['best_weights'])


### Evaluación final de clasificación
Ejecutar tras fijar el modelo usando validación; no ajustar parámetros
en respuesta al test. Test sigue siendo interno a PlantVillage.

In [ ]:
clasificacion = evaluate_test(clasificacion, device=DEVICE)
print('Test clasificación:', clasificacion['test'])


## B. Detección PlantDoc
Entrenamiento independiente a partir de YOLO26n. Este piloto usa dos
clases de enfermedad y recompone los grupos TRAIN/TEST del proveedor:
no constituye el benchmark oficial PlantDoc. La etiqueta genérica
`Potato leaf` se excluye porque no confirma sanidad.

In [ ]:
deteccion = train_experiment(planes['plantdoc'], device=DEVICE)
print('Validación:', deteccion['validation'])
print('Clases:', deteccion['names'])
print('Mejor checkpoint:', deteccion['best_weights'])


### Evaluación final de detección
Para comparar familias, utiliza el cuaderno 03 y su selección por
validación. No reutilices repetidamente este test para elegir modelos.

In [ ]:
deteccion = evaluate_test(deteccion, device=DEVICE)
print('Test detección:', deteccion['test'])


## Entrega al servidor
Cada ejecución conserva `weights/best.pt`, `weights/best.metadata.json`,
`experiment.json`, `args.yaml`, `results.csv` y gráficos que genere Ultralytics.
Se utiliza el directorio real de la ejecución, incluso si se incrementa su nombre.

Para probar esos pesos en el servidor, configura `MODEL_WEIGHTS_PATH` con
`best.pt` y conserva sus metadatos adyacentes. El clasificador no usa SAHI;
el detector de hojas puede usarlo, pero ninguno de estos dos pilotos
produce severidad de lesión ni conteo de insectos para el lazo automático.
El servidor obtiene las clases de `model.names`, no de seis IDs supuestos.